# PC-FNO3D réel (Grand Tunis) — suite de la v3 : plis, cartes, Nabeul (Colab, A100)

1. **Bilan des 3 plis** de validation croisée (`scripts/bilan_plis_grand_tunis.py`) : erreur à la
   station jamais cible, à côté de la v2 et d'ERA5. Règle : la v3 n'est livrée que si les plis
   montrent un gain aux stations jamais vues ; sinon la v2 reste livrée.
2. **Cartes de l'écart v3 − v2** à 10 m et 100 m (`scripts/comparer_modeles_grand_tunis.py`) :
   pas de taches autour des stations ?
3. **Seulement si gain** : v3b = modèle final avec **Nabeul en direction seule**
   (`loss.poids_vitesse_station`), entraîné, évalué, jugé sur les vraies mesures, cartes.

Résultats : `runs/bilan_plis_v3.*`, `runs/comparaisons/`, `runs/grand_tunis_v3b_stations/`.
La session s'arrête à la fin.

## 1. Drive, code et données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")
GT.mkdir(parents=True, exist_ok=True)
JOURNAUX = GT / "journaux"
JOURNAUX.mkdir(exist_ok=True)

# code : archive deposee sur le Drive depuis ines-pc
CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille reelle, forcage ERA5, stations (une archive sur le Drive)
DON = Path("/content/donnees")
if not (DON / "grand_tunis" / "forcage_era5.csv").exists():
    with zipfile.ZipFile(GT / "grand_tunis_donnees.zip") as zf:
        zf.extractall(DON)
# mises a jour deposees apres coup sur le Drive (morceaux de moins de 10 Mo) :
# maj_<fichier> ou maj_<fichier>.000, .001... -> remplacent <fichier> dans donnees/grand_tunis
import collections
morceaux = collections.defaultdict(list)
for f in sorted(GT.glob("maj_*")):
    nom = f.name[4:]
    tete, _, queue = nom.rpartition(".")
    morceaux[tete if queue.isdigit() and tete else nom].append(f)
for nom, fs in morceaux.items():
    with open(DON / "grand_tunis" / nom, "wb") as out:
        for f in sorted(fs):
            out.write(f.read_bytes())
    print("mise a jour :", nom, "<-", [f.name for f in sorted(fs)])
print(sorted(p.name for p in (DON / "grand_tunis").iterdir()))

## 2. Cache (3 parties) et observations

In [ ]:
# cache : les 3 parties du Drive -> disque local ; index fusionne
LOC = Path("/content/grand_tunis")
CACHE, JEU = LOC / "cache", LOC / "jeu"
for k in (1, 2, 3):
    if not (GT / f"cache_partie_{k}.zip").exists():
        raise FileNotFoundError(f"cache_partie_{k}.zip absent : finir le carnet de generation")
if not (CACHE / "index.csv").exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    for k in (1, 2, 3):
        with zipfile.ZipFile(GT / f"cache_partie_{k}.zip") as zf:
            zf.extractall(CACHE)
    with zipfile.ZipFile(GT / "jeu.zip") as zf:
        zf.extractall(JEU)
    import pandas as pd
    idx = pd.concat([pd.read_csv(CACHE / f"index_partie_{k}.csv") for k in (1, 2, 3)])
    idx.drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
import pandas as pd
print(pd.read_csv(CACHE / "index.csv")["split"].value_counts().to_string())

# la grille complete (aneo_grid_points.csv) est dans jeu.zip
if not (GT / "best_dso_v7.pt").exists():
    raise FileNotFoundError("deposer best_dso_v7.pt (poids v7) dans MyDrive/pcfno3d_grand_tunis/")
lancer("controle", "python -u -m pcfno3d -c grand_tunis_v3b_stations_colab check")

# observations reelles du domaine (affinage et jugement)
OBS = Path("/content/obs")
with zipfile.ZipFile(GT / "grand_tunis_obs.zip") as zf:
    zf.extractall(OBS)
print(sorted(p.name for p in OBS.iterdir()))

## 3. Mémoire du GPU

In [ ]:
# memoire : un pas complet (avant + arriere) sur la grille entiere ; recalcul des activations
# couche par couche si le GPU ne suffit pas (memes resultats, ~30 % plus lent)
import torch
from pcfno3d.config import Config
from pcfno3d.geometry import Geometry
from pcfno3d.model import build_model

def essai(recalcul: bool) -> float:
    cfg = Config.load("configs/grand_tunis_v3b_stations_colab.yaml")
    cfg.model.recalcul_activations = recalcul
    geom = Geometry(cfg, device="cuda")
    m = build_model(cfg, geom.c_in).cuda().train()
    x = torch.randn(cfg.train.batch_size, geom.c_in, *geom.shape, device="cuda")
    torch.cuda.reset_peak_memory_stats()
    with torch.autocast("cuda", dtype=torch.bfloat16):
        m(x).float().pow(2).mean().backward()
    pic = torch.cuda.max_memory_allocated() / 1e9
    del m, x
    torch.cuda.empty_cache()
    return pic

total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(torch.cuda.get_device_name(0), f"{total:.0f} Go")
try:
    pic = essai(False)
    RECALCUL = pic > 0.85 * total
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    pic, RECALCUL = float("nan"), True
if RECALCUL:
    pic = essai(True)
print(f"pic {pic:.1f} Go sur {total:.0f} | recalcul des activations : {RECALCUL}")
OPTS = f"--set model.recalcul_activations={str(RECALCUL).lower()}"

## 4. Plis, cartes, v3b si gain, puis fin de la session

In [ ]:
from google.colab import runtime
import traceback
RUNS = GT / "runs"
CARTES = RUNS / "comparaisons"
def essai(nom, cmd):
    try:
        lancer(nom, cmd)
        return 0
    except Exception as e:
        print(e)
        return 1
try:
    # 1. plis : la station jamais cible juge ; code 0 = gain (v3 livrable), 3 = la v2 reste livree
    p = subprocess.run(f"python -u scripts/bilan_plis_grand_tunis.py --runs {RUNS}", shell=True,
                       cwd="/content/PFE")
    gain = p.returncode == 0
    print("GAIN AUX STATIONS JAMAIS VUES :", gain)
    # 2. cartes de l'ecart v3 - v2 a 10 m et 100 m
    essai("cartes_v3", f"python -u scripts/comparer_modeles_grand_tunis.py --conf-a grand_tunis_v2_long_colab "
                       f"--conf-b grand_tunis_v3_stations_colab --obs {OBS} --sortie {CARTES}")
    # 3. v3b (Nabeul en direction seule), seulement si les plis montrent un gain
    if gain:
        conf = "grand_tunis_v3b_stations_colab"
        essai("entrainer_v3b", f"python -u -m pcfno3d -c {conf} {OPTS} train")
        essai("evaluer_v3b", f"python -u -m pcfno3d -c {conf} {OPTS} evaluate")
        essai("valider_v3b", f"python -u scripts/valider_grand_tunis_reel.py -c {conf} --obs {OBS}")
        essai("cartes_v3b", f"python -u scripts/comparer_modeles_grand_tunis.py --conf-a grand_tunis_v2_long_colab "
                            f"--conf-b {conf} --obs {OBS} --sortie {CARTES}")
    else:
        print("pas de gain aux stations jamais vues : la v2 reste le modele livre, pas de v3b")
finally:
    runtime.unassign()